In [2]:
import re
import time
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Load the dataset
df = pd.read_csv("motorcycle_slm_dataset.csv")
print(f"Loaded {len(df)} bikes with {len(df.columns)} attributes.")

# 2. Generate comprehensive Question-Answer pairs from every row & column
def build_qa_pairs(df):
    pairs = []
    for _, r in df.iterrows():
        bike_id = str(r["bike_id"]).strip()
        brand = str(r["brand"]).strip()
        model = str(r["model"]).strip()
        full_name = f"{brand} {model}"
        cat = str(r["category"]).strip()
        cc = int(r["engine_cc"])
        cyl = int(r["cylinders"])
        cool = str(r["cooling_system"]).strip()
        power = r["max_power_bhp"]
        torque = r["max_torque_nm"]
        trans = str(r["transmission"]).strip()
        fuel = r["fuel_capacity_l"]
        mileage = r["mileage_kmpl"]
        weight = int(r["kerb_weight_kg"])
        seat = int(r["seat_height_mm"])
        abs_t = str(r["abs_type"]).strip()
        price = int(r["ex_showroom_price_inr"])
        desc = str(r["description"]).strip()
        pros = str(r["pros"]).strip()
        cons = str(r["cons"]).strip()

        engine_str = "an Electric Motor (0cc)" if cc == 0 else f"a {cc}cc, {cyl}-cylinder, {cool} engine"
        mileage_unit = "km/charge" if cc == 0 else "kmpl"
        fuel_str = f"Battery capacity: {fuel} kWh" if cc == 0 else f"Fuel tank capacity: {fuel} liters"

        # A. Full Overview
        overview = (
            f"{full_name} ({bike_id}) is a {cat} motorcycle priced at Rs. {price:,} (ex-showroom). "
            f"It is powered by {engine_str} producing {power} bhp and {torque} Nm of torque with a {trans} transmission. "
            f"Mileage: {mileage} {mileage_unit}, Kerb Weight: {weight} kg, Seat Height: {seat} mm, Braking: {abs_t}. {desc}"
        )
        for q in [
            f"Tell me about {full_name}",
            f"What is {full_name}",
            f"Details of {full_name}",
            f"{full_name}",
            f"Tell me about {model}",
            f"{model}"
        ]:
            pairs.append((q, overview))

        # B. Price
        ans_price = f"The ex-showroom price of {full_name} is Rs. {price:,}."
        for q in [
            f"What is the price of {full_name}",
            f"Price of {full_name}",
            f"How much does {full_name} cost",
            f"Price of {model}",
            f"{full_name} price"
        ]:
            pairs.append((q, ans_price))

        # C. Engine & Performance Specs
        ans_eng = f"{full_name} features {engine_str} generating {power} bhp of power and {torque} Nm of torque, paired with a {trans} gearbox."
        for q in [
            f"Engine specs of {full_name}",
            f"What is the engine of {full_name}",
            f"Specs of {full_name}",
            f"Specifications of {full_name}",
            f"Power and torque of {full_name}",
            f"Specs of {model}"
        ]:
            pairs.append((q, ans_eng))

        # D. Mileage & Fuel/Battery Capacity
        ans_mil = f"{full_name} delivers a claimed mileage of {mileage} {mileage_unit}. {fuel_str}."
        for q in [
            f"What is the mileage of {full_name}",
            f"Mileage of {full_name}",
            f"Fuel capacity of {full_name}",
            f"Mileage of {model}",
            f"{full_name} mileage"
        ]:
            pairs.append((q, ans_mil))

        # E. Dimensions, Weight, Seat Height & ABS
        ans_dim = f"{full_name} is a {cat} bike with a kerb weight of {weight} kg, seat height of {seat} mm, and {abs_t} braking."
        for q in [
            f"Weight and seat height of {full_name}",
            f"Seat height of {full_name}",
            f"Weight of {full_name}",
            f"ABS of {full_name}",
            f"Category of {full_name}"
        ]:
            pairs.append((q, ans_dim))

        # F. Pros & Cons
        ans_pc = f"Pros of {full_name}: {pros} Cons of {full_name}: {cons}"
        for q in [
            f"Pros and cons of {full_name}",
            f"Pros of {full_name}",
            f"Cons of {full_name}",
            f"Review of {full_name}",
            f"Pros and cons of {model}"
        ]:
            pairs.append((q, ans_pc))

    # G. Dataset-wide Comparative & Aggregate Queries
    pairs.append(("How many bikes are in the dataset", f"There are {len(df)} motorcycles and scooters in the dataset across {df['brand'].nunique()} brands."))

    cheapest = df.loc[df["ex_showroom_price_inr"].idxmin()]
    expensive = df.loc[df["ex_showroom_price_inr"].idxmax()]
    powerful = df.loc[df["max_power_bhp"].idxmax()]
    petrol_df = df[df["engine_cc"] > 0]
    hi_mil = petrol_df.loc[petrol_df["mileage_kmpl"].idxmax()]

    for q in ["Cheapest bike", "What is the cheapest bike"]:
        pairs.append((q, f"The cheapest bike in the dataset is {cheapest['brand']} {cheapest['model']} at Rs. {int(cheapest['ex_showroom_price_inr']):,}."))
    for q in ["Most expensive bike", "What is the most expensive bike"]:
        pairs.append((q, f"The most expensive bike in the dataset is {expensive['brand']} {expensive['model']} at Rs. {int(expensive['ex_showroom_price_inr']):,}."))
    for q in ["Most powerful bike", "What is the most powerful bike"]:
        pairs.append((q, f"The most powerful bike in the dataset is {powerful['brand']} {powerful['model']} with {powerful['max_power_bhp']} bhp."))
    for q in ["Highest mileage bike", "What is the highest mileage bike"]:
        pairs.append((q, f"The highest mileage petrol bike in the dataset is {hi_mil['brand']} {hi_mil['model']} with {hi_mil['mileage_kmpl']} kmpl."))

    for brand, grp in df.groupby("brand"):
        models = ", ".join(grp["model"].tolist())
        pairs.append((f"List all {brand} bikes", f"{brand} bikes in the dataset: {models}."))
        pairs.append((f"{brand} bikes", f"{brand} bikes in the dataset: {models}."))

    for cat, grp in df.groupby("category"):
        names = ", ".join((grp["brand"] + " " + grp["model"]).tolist())
        pairs.append((f"List all {cat} bikes", f"{cat} models in the dataset: {names}."))

    return pairs

qa_pairs = build_qa_pairs(df)
print(f"Generated {len(qa_pairs)} structured Q&A training examples.")

Loaded 40 bikes with 19 attributes.
Generated 1344 structured Q&A training examples.


Tokenizer, MiniGPT Transformer Architecture & Training Loop

In [3]:
# 1. Word + Punctuation Tokenizer
def tokenize(text):
    text = text.strip().lower()
    return re.findall(r"[a-z0-9]+(?:[.,'-][a-z0-9]+)*|[^\w\s]", text)

SPECIAL_TOKENS = ["<PAD>", "<BOS>", "<SEP>", "<EOS>", "<UNK>"]
vocab_words = set()
for q, a in qa_pairs:
    vocab_words.update(tokenize(q))
    vocab_words.update(tokenize(a))

vocab = SPECIAL_TOKENS + sorted(vocab_words)
stoi = {tok: idx for idx, tok in enumerate(vocab)}
itos = {idx: tok for idx, tok in enumerate(vocab)}

PAD_ID = stoi["<PAD>"]
BOS_ID = stoi["<BOS>"]
SEP_ID = stoi["<SEP>"]
EOS_ID = stoi["<EOS>"]
UNK_ID = stoi["<UNK>"]
BLOCK_SIZE = 128

# Encode all Q&A pairs as: <BOS> question_tokens <SEP> answer_tokens <EOS> <PAD>...
encoded_data = []
for q, a in qa_pairs:
    seq = [BOS_ID] + [stoi[t] for t in tokenize(q)] + [SEP_ID] + [stoi[t] for t in tokenize(a)] + [EOS_ID]
    if len(seq) > BLOCK_SIZE:
        seq = seq[:BLOCK_SIZE - 1] + [EOS_ID]
    else:
        seq = seq + [PAD_ID] * (BLOCK_SIZE - len(seq))
    encoded_data.append(seq)

data_tensor = torch.tensor(encoded_data, dtype=torch.long)
print(f"Vocabulary Size: {len(vocab)} tokens | Context Window: {BLOCK_SIZE}")

# 2. MiniGPT Decoder-Only Causal Transformer Architecture
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        )

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        att = att.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class BikeMiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=128, n_head=4, n_layer=3, block_size=128):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([TransformerBlock(n_embd, n_head, block_size) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.tok_emb.weight = self.lm_head.weight  # Weight tying

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=PAD_ID)
        return logits, loss

    @torch.no_grad()
    def generate_answer(self, question, max_new_tokens=95):
        self.eval()
        device = next(self.parameters()).device
        clean_q = question.strip().rstrip("?!.")
        q_ids = [stoi.get(t, UNK_ID) for t in tokenize(clean_q)]
        idx = torch.tensor([[BOS_ID] + q_ids + [SEP_ID]], dtype=torch.long, device=device)

        generated = []
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)  # Greedy decoding for factual accuracy
            tok_id = next_id.item()
            if tok_id == EOS_ID:
                break
            generated.append(itos[tok_id])
            idx = torch.cat([idx, next_id], dim=1)

        text = " ".join(generated)
        text = re.sub(r"\s+([.,;:!?)\]])", r"\1", text)
        text = re.sub(r"([(\[])\s+", r"\1", text)
        return text

# 3. Train the MiniGPT Model
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = BikeMiniGPT(len(vocab), n_embd=128, n_head=4, n_layer=3, block_size=BLOCK_SIZE).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Training BikeMiniGPT on {device.upper()} | Total Parameters: {total_params:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=35)
data_tensor = data_tensor.to(device)

start_time = time.time()
EPOCHS = 35
BATCH_SIZE = 64

for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(data_tensor.size(0), device=device)
    total_loss = 0.0
    batches = 0
    for i in range(0, data_tensor.size(0), BATCH_SIZE):
        batch = data_tensor[perm[i:i + BATCH_SIZE]]
        x, y = batch[:, :-1], batch[:, 1:]
        optimizer.zero_grad()
        _, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        batches += 1
    scheduler.step()
    if epoch in [1, 10, 20, 30, 35]:
        print(f"Epoch {epoch:02d}/{EPOCHS} | Loss: {total_loss / batches:.4f} | Time: {time.time() - start_time:.1f}s")

# Save trained model checkpoint for GitHub submission
torch.save({
    "model_state": model.state_dict(),
    "stoi": stoi,
    "itos": itos,
    "config": {"vocab_size": len(vocab), "n_embd": 128, "n_head": 4, "n_layer": 3, "block_size": BLOCK_SIZE}
}, "bike_minigpt.pt")
print("Saved trained weights to bike_minigpt.pt")

Vocabulary Size: 1210 tokens | Context Window: 128
Training BikeMiniGPT on CPU | Total Parameters: 766,336
Epoch 01/35 | Loss: 5.3059 | Time: 20.0s
Epoch 10/35 | Loss: 0.7297 | Time: 206.2s
Epoch 20/35 | Loss: 0.1722 | Time: 409.9s
Epoch 30/35 | Loss: 0.1546 | Time: 617.3s
Epoch 35/35 | Loss: 0.1534 | Time: 720.3s
Saved trained weights to bike_minigpt.pt


In [4]:
import ipywidgets as widgets
from IPython.display import display, HTML

header = HTML("<h3>🏍️ Bike-MiniGPT Small Language Model (Interactive Demo)</h3><p>Ask any question about the 40 motorcycles & scooters in the dataset:</p>")

input_box = widgets.Text(
    value="Tell me about Royal Enfield Classic 350",
    placeholder="e.g., Price of KTM Duke 390 | Pros and cons of Yamaha YZF R15 V4 | Cheapest bike",
    description="Your Question:",
    layout=widgets.Layout(width="75%"),
    style={"description_width": "110px"}
)

ask_button = widgets.Button(
    description="Ask MiniGPT",
    button_style="primary",
    icon="rocket"
)

sample_prompts = [
    "Tell me about Royal Enfield Classic 350",
    "What is the price of KTM Duke 390",
    "Pros and cons of Yamaha YZF R15 V4",
    "What is the mileage of Hero Splendor Plus",
    "Most powerful bike",
    "List all Royal Enfield bikes"
]

sample_buttons = [
    widgets.Button(description=p, layout=widgets.Layout(width="auto"), button_style="info")
    for p in sample_prompts
]

output_area = widgets.Output()

def run_query(_=None):
    query = input_box.value.strip()
    if not query:
        return
    answer = model.generate_answer(query)
    with output_area:
        output_area.clear_output()
        display(HTML(f"""
        <div style="border-left: 4px solid #1a73e8; background: #f8f9fa; padding: 12px; margin-top: 10px; border-radius: 4px; color: #111;">
            <b>Prompt:</b> {query}<br><br>
            <b>MiniGPT Response:</b> {answer}
        </div>
        """))

def on_sample_click(btn):
    input_box.value = btn.description
    run_query()

ask_button.on_click(run_query)
input_box.on_submit(run_query)
for b in sample_buttons:
    b.on_click(on_sample_click)

display(header)
display(widgets.HBox([input_box, ask_button]))
display(HTML("<small><b>Quick Sample Prompts:</b></small>"))
display(widgets.HBox(sample_buttons[:3]))
display(widgets.HBox(sample_buttons[3:]))
display(output_area)
run_query()

Output()